In [1]:
import numpy as np
import pandas as pd
import ast
from gensim.utils import simple_preprocess
from sentence_transformers import util
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

In [22]:
df = pd.read_csv('5000_movies.csv')
df.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [23]:
df2 = pd.read_csv('5000_movies_credits.csv')
df2.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [24]:
df = df.merge(df2, on='title')
df.shape

(4809, 23)

In [25]:
#important features
# genres, id, keywords, original_title, overview, cast, crew

In [26]:
df = df[['id', 'original_title', 'overview', 'genres', 'keywords', 'cast', 'crew', 'vote_count', 'vote_average', 'popularity', 'release_date']]
df

,id,original_title,overview,genres,keywords,cast,crew,vote_count,vote_average,popularity,release_date
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",11800,7.2,150.437577,2009-12-10
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",4500,6.9,139.082615,2007-05-19
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",4466,6.3,107.376788,2015-10-26
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",9106,7.6,112.312950,2012-07-16
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",2124,6.1,43.926995,2012-03-07
...,...,...,...,...,...,...,...,...,...,...,...
4804,9367,El Mariachi,El Mariachi just wants to play his guitar and ...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 5616, ""name"": ""united states\u2013mexi...","[{""cast_id"": 1, ""character"": ""El Mariachi"", ""c...","[{""credit_id"": ""52fe44eec3a36847f80b280b"", ""de...",238,6.6,14.269792,1992-09-04
4805,72766,Newlyweds,A newlywed couple's honeymoon is upended by th...,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 10749, ""...",[],"[{""cast_id"": 1, ""character"": ""Buzzy"", ""credit_...","[{""credit_id"": ""52fe487dc3a368484e0fb013"", ""de...",5,5.9,0.642552,2011-12-26
4806,231617,"Signed, Sealed, Delivered","""Signed, Sealed, Delivered"" introduces a dedic...","[{""id"": 35, ""name"": ""Comedy""}, {""id"": 18, ""nam...","[{""id"": 248, ""name"": ""date""}, {""id"": 699, ""nam...","[{""cast_id"": 8, ""character"": ""Oliver O\u2019To...","[{""credit_id"": ""52fe4df3c3a36847f8275ecf"", ""de...",6,7.0,1.444476,2013-10-13
4807,126186,Shanghai Calling,When ambitious New York attorney Sam is sent t...,[],[],"[{""cast_id"": 3, ""character"": ""Sam"", ""credit_id...","[{""credit_id"": ""52fe4ad9c3a368484e16a36b"", ""de...",7,5.7,0.857008,2012-05-03


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              4809 non-null   int64  
 1   original_title  4809 non-null   object 
 2   overview        4806 non-null   object 
 3   genres          4809 non-null   object 
 4   keywords        4809 non-null   object 
 5   cast            4809 non-null   object 
 6   crew            4809 non-null   object 
 7   vote_count      4809 non-null   int64  
 8   vote_average    4809 non-null   float64
 9   popularity      4809 non-null   float64
 10  release_date    4808 non-null   object 
dtypes: float64(2), int64(2), object(7)
memory usage: 413.4+ KB


In [28]:
# checking null values

In [29]:
df.isnull().sum()

id                0
original_title    0
overview          3
genres            0
keywords          0
cast              0
crew              0
vote_count        0
vote_average      0
popularity        0
release_date      1
dtype: int64

In [30]:
df.dropna(axis=0, inplace=True)

In [31]:
# checking duplicates values

In [32]:
df.duplicated().sum()

0

# text preprocessing

In [33]:
# extract text from geners, keywords columns
def extract_text(obj):
    l = []
    for i in ast.literal_eval(obj):
        l.append(i['name'])
    return l

In [34]:
df['genres'] = df['genres'].apply(extract_text)
df['keywords'] = df['keywords'].apply(extract_text)

In [38]:
# extract text from cast column
def extract_text2(obj):
    l = []
    count = 0
    for i in ast.literal_eval(obj):
        if count == 3:
            break
        l.append(i['name'])
        count += 1
        
    return l

In [39]:
df['cast'] = df['cast'].apply(extract_text2)

In [50]:
# extract text from crew column
def extract_director(obj):
    l = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            l.append(i['name'])
            break
    return l

In [51]:
df['crew'] = df['crew'].apply(extract_director)

In [52]:
df['overview'] = df['overview'].apply(lambda x: x.split())

In [53]:
df

,id,original_title,overview,genres,keywords,cast,crew,vote_count,vote_average,popularity,release_date
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron],11800,7.2,150.437577,2009-12-10
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski],4500,6.9,139.082615,2007-05-19
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes],4466,6.3,107.376788,2015-10-26
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan],9106,7.6,112.312950,2012-07-16
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton],2124,6.1,43.926995,2012-03-07
...,...,...,...,...,...,...,...,...,...,...,...
4804,9367,El Mariachi,"[El, Mariachi, just, wants, to, play, his, gui...","[Action, Crime, Thriller]","[united states–mexico barrier, legs, arms, pap...","[Carlos Gallardo, Jaime de Hoyos, Peter Marqua...",[Robert Rodriguez],238,6.6,14.269792,1992-09-04
4805,72766,Newlyweds,"[A, newlywed, couple's, honeymoon, is, upended...","[Comedy, Romance]",[],"[Edward Burns, Kerry Bishé, Marsha Dietlein]",[Edward Burns],5,5.9,0.642552,2011-12-26
4806,231617,"Signed, Sealed, Delivered","[""Signed,, Sealed,, Delivered"", introduces, a,...","[Comedy, Drama, Romance, TV Movie]","[date, love at first sight, narration, investi...","[Eric Mabius, Kristin Booth, Crystal Lowe]",[Scott Smith],6,7.0,1.444476,2013-10-13
4807,126186,Shanghai Calling,"[When, ambitious, New, York, attorney, Sam, is...",[],[],"[Daniel Henney, Eliza Coupe, Bill Paxton]",[Daniel Hsia],7,5.7,0.857008,2012-05-03


In [54]:
# remove spaces 
df['genres'] = df['genres'].apply(lambda x: [i.replace(' ', '') for i in x])
df['keywords'] = df['keywords'].apply(lambda x: [i.replace(' ', '') for i in x])
df['cast'] = df['cast'].apply(lambda x: [i.replace(' ', '') for i in x])
df['crew'] = df['crew'].apply(lambda x: [i.replace(' ', '') for i in x])

In [55]:
df.head()

,id,original_title,overview,genres,keywords,cast,crew,vote_count,vote_average,popularity,release_date
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],11800,7.2,150.437577,2009-12-10
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],4500,6.9,139.082615,2007-05-19
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes],4466,6.3,107.376788,2015-10-26
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan],9106,7.6,112.312950,2012-07-16
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton],2124,6.1,43.926995,2012-03-07


In [56]:
df['movie_text'] = df['overview'] + df['genres'] + df['keywords'] + df['cast'] + df['crew']
df

,id,original_title,overview,genres,keywords,cast,crew,vote_count,vote_average,popularity,release_date,movie_text
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],11800,7.2,150.437577,2009-12-10,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],4500,6.9,139.082615,2007-05-19,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes],4466,6.3,107.376788,2015-10-26,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan],9106,7.6,112.312950,2012-07-16,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton],2124,6.1,43.926995,2012-03-07,"[John, Carter, is, a, war-weary,, former, mili..."
...,...,...,...,...,...,...,...,...,...,...,...,...
4804,9367,El Mariachi,"[El, Mariachi, just, wants, to, play, his, gui...","[Action, Crime, Thriller]","[unitedstates–mexicobarrier, legs, arms, paper...","[CarlosGallardo, JaimedeHoyos, PeterMarquardt]",[RobertRodriguez],238,6.6,14.269792,1992-09-04,"[El, Mariachi, just, wants, to, play, his, gui..."
4805,72766,Newlyweds,"[A, newlywed, couple's, honeymoon, is, upended...","[Comedy, Romance]",[],"[EdwardBurns, KerryBishé, MarshaDietlein]",[EdwardBurns],5,5.9,0.642552,2011-12-26,"[A, newlywed, couple's, honeymoon, is, upended..."
4806,231617,"Signed, Sealed, Delivered","[""Signed,, Sealed,, Delivered"", introduces, a,...","[Comedy, Drama, Romance, TVMovie]","[date, loveatfirstsight, narration, investigat...","[EricMabius, KristinBooth, CrystalLowe]",[ScottSmith],6,7.0,1.444476,2013-10-13,"[""Signed,, Sealed,, Delivered"", introduces, a,..."
4807,126186,Shanghai Calling,"[When, ambitious, New, York, attorney, Sam, is...",[],[],"[DanielHenney, ElizaCoupe, BillPaxton]",[DanielHsia],7,5.7,0.857008,2012-05-03,"[When, ambitious, New, York, attorney, Sam, is..."


In [57]:
# create new dataframe
new_df = df[['id', 'original_title', 'movie_text']]
new_df

,id,original_title,movie_text
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili..."
...,...,...,...
4804,9367,El Mariachi,"[El, Mariachi, just, wants, to, play, his, gui..."
4805,72766,Newlyweds,"[A, newlywed, couple's, honeymoon, is, upended..."
4806,231617,"Signed, Sealed, Delivered","[""Signed,, Sealed,, Delivered"", introduces, a,..."
4807,126186,Shanghai Calling,"[When, ambitious, New, York, attorney, Sam, is..."


In [58]:
new_df['movie_text'] =  new_df['movie_text'].apply(lambda x: ' '.join(x))

In [59]:
new_df['movie_text'] = new_df['movie_text'].str.lower()
new_df['date'] = df['release_date']
new_df['rating'] = df['vote_average']

In [60]:
new_df.head(2)

,id,original_title,movie_text,date,rating
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di...",2009-12-10,7.2
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha...",2007-05-19,6.9


# popularity based

new dataframe for popular movies

In [61]:
popular_df = df[['id', 'original_title', 'vote_count', 'vote_average', 'popularity', 'release_date']]
popular_df.rename(columns={'release_date' : 'date'}, inplace=True)
popular_df

,id,original_title,vote_count,vote_average,popularity,date
0,19995,Avatar,11800,7.2,150.437577,2009-12-10
1,285,Pirates of the Caribbean: At World's End,4500,6.9,139.082615,2007-05-19
2,206647,Spectre,4466,6.3,107.376788,2015-10-26
3,49026,The Dark Knight Rises,9106,7.6,112.312950,2012-07-16
4,49529,John Carter,2124,6.1,43.926995,2012-03-07
...,...,...,...,...,...,...
4804,9367,El Mariachi,238,6.6,14.269792,1992-09-04
4805,72766,Newlyweds,5,5.9,0.642552,2011-12-26
4806,231617,"Signed, Sealed, Delivered",6,7.0,1.444476,2013-10-13
4807,126186,Shanghai Calling,7,5.7,0.857008,2012-05-03


# trending movie

In [68]:
trending_movie = popular_df[popular_df['popularity'] > 150]

In [72]:
trending_movie = trending_movie[(trending_movie['vote_average'] > 7) & (trending_movie['vote_count'] > 5000)].sort_values('popularity', ascending=False)

In [75]:
trending_movie

,id,original_title,vote_count,vote_average,popularity,date
95,157336,Interstellar,10867,8.1,724.247784,2014-11-05
788,293660,Deadpool,10995,7.4,514.569956,2016-02-09
94,118340,Guardians of the Galaxy,9742,7.9,481.098624,2014-07-30
127,76341,Mad Max: Fury Road,9427,7.2,434.278564,2015-05-13
199,22,Pirates of the Caribbean: The Curse of the Bla...,6985,7.5,271.972889,2003-07-09
88,177572,Big Hero 6,6135,7.8,203.734590,2014-10-24
26,271110,Captain America: Civil War,7241,7.1,198.372395,2016-04-27
65,155,The Dark Knight,12002,8.2,187.322927,2008-07-16
270,286217,The Martian,7268,7.6,167.932870,2015-09-30
96,27205,Inception,13752,8.1,167.583710,2010-07-14


# high rated movie

In [76]:
high_rated_movie = popular_df[popular_df['vote_count'] > 5000].sort_values('vote_average', ascending=False)

In [78]:
high_rated_movie = high_rated_movie[high_rated_movie['vote_average'] > 7.5].sort_values(['vote_count'], ascending=False).head(25)

In [79]:
high_rated_movie

,id,original_title,vote_count,vote_average,popularity,date
96,27205,Inception,13752,8.1,167.583710,2010-07-14
65,155,The Dark Knight,12002,8.2,187.322927,2008-07-16
95,157336,Interstellar,10867,8.1,724.247784,2014-11-05
287,68718,Django Unchained,10099,7.8,82.121691,2012-12-25
94,118340,Guardians of the Galaxy,9742,7.9,481.098624,2014-07-30
662,550,Fight Club,9413,8.3,146.757391,1999-10-15
3,49026,The Dark Knight Rises,9106,7.6,112.312950,2012-07-16
634,603,The Matrix,8907,7.9,104.309993,1999-03-30
262,120,The Lord of the Rings: The Fellowship of the Ring,8705,8.0,138.049577,2001-12-18
3235,680,Pulp Fiction,8428,8.3,121.463076,1994-10-08


# content based

In [80]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

In [81]:
# Step 1: Load embeddings model
model = SentenceTransformer("all-MiniLM-L6-v2")

In [82]:
new_df = new_df.reset_index(drop=True)
new_df

,id,original_title,movie_text,date,rating
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di...",2009-12-10,7.2
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha...",2007-05-19,6.9
2,206647,Spectre,a cryptic message from bond’s past sends him o...,2015-10-26,6.3
3,49026,The Dark Knight Rises,following the death of district attorney harve...,2012-07-16,7.6
4,49529,John Carter,"john carter is a war-weary, former military ca...",2012-03-07,6.1
...,...,...,...,...,...
4800,9367,El Mariachi,el mariachi just wants to play his guitar and ...,1992-09-04,6.6
4801,72766,Newlyweds,a newlywed couple's honeymoon is upended by th...,2011-12-26,5.9
4802,231617,"Signed, Sealed, Delivered","""signed, sealed, delivered"" introduces a dedic...",2013-10-13,7.0
4803,126186,Shanghai Calling,when ambitious new york attorney sam is sent t...,2012-05-03,5.7


In [83]:
# generate embedding for movie text
embedding = model.encode(new_df['movie_text'])
embedding = np.array(embedding).astype('float32')

In [84]:
faiss.normalize_L2(embedding)
embedding_dim = embedding.shape[1]
movie_vector_database = faiss.IndexFlatIP(embedding_dim)
movie_vector_database.add(embedding)

# searching throught topics

In [88]:
def search_by_topics():
    # User input
    user_topic = input('Enter your topic : ')
    # Encode user input
    user_topic_embedding = model.encode([user_topic]).astype("float32")
    
    # Search top 3 similar movies
    D, I = movie_vector_database.search(user_topic_embedding, k=5)
    
    print("Recommended Movies:")
    for idx in I[0]:
        print(new_df['original_title'][idx])

In [107]:
search_by_topics()

Enter your topic :  horror movies


Recommended Movies:
Grindhouse
Scary Movie 2
Book of Shadows: Blair Witch 2
Disaster Movie
The Horror Network Vol. 1


# searching through movies

In [117]:
# generate embedding for movie text
movie_embedding = model.encode(new_df['original_title'])
movie_embedding = np.array(movie_embedding).astype('float32')

In [118]:
faiss.normalize_L2(movie_embedding)
embedding_dim = embedding.shape[1]
title_vector_database = faiss.IndexFlatIP(embedding_dim)
title_vector_database.add(movie_embedding)

In [140]:
def recommend_by_movie_name(movie_name, top_k=5):
    # Check if movie exists
    if movie_name not in new_df['original_title'].values:
        print(f"Movie '{movie_name}' not found in database!")
        print('You mean these movies : \n')
        embedding = model.encode([movie_name]).astype('float32')
        d,i = title_vector_database.search(embedding, top_k)
        l = []
        for idx in i[0]:
            l.append(new_df['original_title'][idx])
        return l
    
    # Get description of input movie
    movie_description = new_df[new_df['original_title'] == movie_name]['movie_text'].values[0]
    # Encode description
    movie_emb = model.encode([movie_description]).astype("float32")
    #Search top-k similar movies
    D, I = movie_vector_database.search(movie_emb, top_k + 1) 
    
    recommended_titles = []
    for idx in I[0]:
        if new_df['original_title'][idx] != movie_name:
            recommended_titles.append(new_df['original_title'][idx])
        if len(recommended_titles) >= top_k:
            break
    
    return recommended_titles

In [148]:
recommend_by_movie_name('Toy Story')

['Toy Story 2', 'Toy Story 3', 'Envy', 'St. Vincent', 'The Way Way Back']

In [149]:
recommend_by_movie_name('iron')

Movie 'iron' not found in database!
You mean these movies : 



['Iron Man', 'Ironclad', 'Steel', 'The Iron Lady', 'Rust']

# load tmdb poster 

In [43]:
import requests

In [44]:
def fetch_poster_url(movie_id):
    try:
        url = f'https://api.themoviedb.org/3/movie/{movie_id}?api_key=9752955cc16c403773d98d6907ede59d'
        data = requests.get(url, timeout=3).json()
        poster_path = data['poster_path']
        return 'https://image.tmdb.org/t/p/185/' + poster_path
    except:
        return 'https://via.placeholder/com/500*750?text=No+Image'


In [45]:
new_df['poster_url'] = new_df['id'].apply(fetch_poster_url)

In [83]:
trending_movie['poster_url'] = trending_movie['id'].apply(fetch_poster_url)

In [84]:
high_rated_movie['poster_url'] = high_rated_movie['id'].apply(fetch_poster_url)

In [80]:
import pickle

In [89]:
# pickle.dump(new_df, open('movie.pkl', 'wb'))
pickle.dump(trending_movie, open('trending_movie.pkl', 'wb'))
pickle.dump(high_rated_movie, open('high_rated_movie.pkl', 'wb'))
# pickle.dump(movie_vector_database, open('movie_vector_database.pkl', 'wb'))

In [82]:
model.save_pretrained('./content_movie_model')